# Edge sensitivity and Shapley results

This notebook reads the fixed-weight edge ablation and sampled Shapley outputs for the `replace_000` and `replace_100` models.

- **Edge removal**: disables one existing C?C connection, with the trained parameters held fixed. The drop is baseline metric minus the metric after removal.
- **Shapley**: averages each edge's marginal change in the chosen metric over sampled edge orderings.
- **Retraining selections**: ranks validation-set removals using `SELECTION_METRIC`, then writes the top N edges for each model to `selected_edges_for_retraining.csv`. The final test split is displayed for reporting, not used to choose edges.

Change `DATASET` to `celeba` or `cfmnist` and adjust `SELECTION_METRIC` below.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists() and (PROJECT_ROOT.parent / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET = "celeba"  # "celeba" or "cfmnist"
ANALYSIS_CONFIG = PROJECT_ROOT / "all_configs" / "analysis" / "edge_sensitivity" / f"{DATASET}_replace_000_vs_replace_100.yaml"
analysis_config = yaml.safe_load(ANALYSIS_CONFIG.read_text(encoding="utf-8"))
SELECTION_METRIC = analysis_config.get("selection_metric", "task_accuracy")  # can override here, e.g. concept_accuracy_003
TOP_N = int(analysis_config.get("top_edges_per_model", 3))
RESULTS_DIR = PROJECT_ROOT / "experiments" / "edge_sensitivity" / DATASET
SELECTION_CSV = RESULTS_DIR / "selected_edges_for_retraining.csv"

coalitions = pd.read_csv(RESULTS_DIR / "coalition_metrics.csv")
shapley = pd.read_csv(RESULTS_DIR / "shapley_values.csv")
edges = pd.read_csv(RESULTS_DIR / "edge_list.csv")
print("Loaded", DATASET, "from", RESULTS_DIR)
print("Models:", sorted(edges.model.unique()))
print("Available metric examples:", sorted(coalitions.metric.unique())[:30])

## 1. Baseline predictions and intervention performance

The primary measures are task accuracy, overall concept accuracy, individual concept accuracies, and intervention accuracy. The intervention curves report task and concept accuracy as the number of corrected concepts/groups changes. The AUC metrics summarize each curve over the normalized intervention budget.

In [ ]:
baseline = coalitions[(coalitions.coalition_type == "baseline") & (coalitions.split.isin(["val", "test"]))].copy()
base_pivot = baseline.pivot_table(index=["metric"], columns=["model", "split"], values="metric_value")
display(base_pivot.round(4))

# Compare the ordinary and intervention task-accuracy curves on test.
curve = baseline[baseline.metric.str.startswith("intervention_task_accuracy_") & (baseline.split == "test")].copy()
if not curve.empty:
    parts = curve.metric.str.extract(r"intervention_task_accuracy_(individual|group)_(\d+)")
    curve["intervention_mode"] = parts[0]
    curve["num_interventions"] = parts[1].astype(int)
    for mode, frame in curve.groupby("intervention_mode"):
        ax = frame.pivot(index="num_interventions", columns="model", values="metric_value").sort_index().plot(marker="o", figsize=(7,4), title=f"Test intervention task accuracy: {mode}")
        ax.set_ylabel("Task accuracy"); ax.set_ylim(0,1); plt.tight_layout(); plt.show()

## 2. Which existing edges matter when weights are frozen?

Each row below is the change from the complete graph to a model with one edge removed. Positive drops mean the removal hurt that metric; negative values mean the metric improved after removal. Validation drops are used to choose edges for retraining.

In [ ]:
base = coalitions[(coalitions.coalition_type == "baseline")].set_index(["model", "split", "metric"]).metric_value
abl = coalitions[coalitions.coalition_type == "single_edge_removed"].copy()
abl["baseline_value"] = [base.loc[(r.model, r.split, r.metric)] for r in abl.itertuples()]
abl["metric_drop_from_removal"] = abl.baseline_value - abl.metric_value
edge_effects = abl.copy()
validation_effects = edge_effects[edge_effects.split == "val"].copy()

# Human-readable edge names use the graph's source and target concept labels.
validation_effects["edge_label"] = (
    validation_effects.source_concept.astype(str).str.replace("_", " ", regex=False)
    + " ? "
    + validation_effects.target_concept.astype(str).str.replace("_", " ", regex=False)
)
selected_view = validation_effects[validation_effects.metric == SELECTION_METRIC].sort_values(
    "metric_drop_from_removal", ascending=False
)
for model, frame in selected_view.groupby("model"):
    print(f"{model} ranked by validation {SELECTION_METRIC}")
    display(frame[["edge_label", "baseline_value", "metric_value", "metric_drop_from_removal"]]
            .head(15).round(4))

# One figure: Graph A on top and Graph B below. Each heatmap contains every
# validation metric recorded for each edge. Positive values mean removing the
# edge lowered the metric; negative values mean removal improved it.
model_order = [m for m in ["replace_000", "replace_100"] if m in validation_effects.model.unique()]
metric_order = list(validation_effects.metric.drop_duplicates())
metric_order = sorted(metric_order, key=lambda m: (
    0 if m == "task_accuracy" else
    1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else
    5,
    m
))
edge_labels_by_model = {}
for model in model_order:
    edge_labels_by_model[model] = (validation_effects[validation_effects.model == model]
                                   [["edge_label", "source_index", "target_index"]]
                                   .drop_duplicates()
                                   .sort_values(["source_index", "target_index"])
                                   .edge_label.tolist())

all_drops = validation_effects.metric_drop_from_removal.to_numpy(dtype=float)
color_limit = float(np.nanmax(np.abs(all_drops))) if len(all_drops) else 1.0
color_limit = color_limit or 1.0
fig_height = max(7, 0.30 * len(metric_order) * len(model_order))
fig, axes = plt.subplots(len(model_order), 1, figsize=(max(12, 0.85 * max(map(len, edge_labels_by_model.values()))), fig_height), squeeze=False)
image = None
for ax, model in zip(axes[:, 0], model_order):
    frame = validation_effects[validation_effects.model == model]
    edge_order = edge_labels_by_model[model]
    matrix = (frame.pivot_table(index="metric", columns="edge_label", values="metric_drop_from_removal", aggfunc="mean")
              .reindex(index=metric_order, columns=edge_order))
    image = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap="RdBu_r",
                      vmin=-color_limit, vmax=color_limit, interpolation="nearest")
    ax.set_title(f"{'Graph A' if model == 'replace_000' else 'Graph B'} ({model})", loc="left", fontweight="bold")
    ax.set_xticks(np.arange(len(edge_order)))
    ax.set_xticklabels(edge_order, rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels(metric_order)
    ax.set_xlabel("Removed edge (source concept ? target concept)")
    ax.set_ylabel("Validation metric")
    # Print the numeric effect in every cell for direct inspection.
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iat[i, j]
            if pd.notna(value):
                text_color = "white" if abs(value) > color_limit * 0.48 else "black"
                ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=7, color=text_color)
fig.colorbar(image, ax=axes[:, 0].tolist(), label="Baseline ? after edge removal (positive = removal hurts)", shrink=0.85, pad=0.02)
fig.suptitle("Validation edge sensitivity across all recorded metrics", y=1.002)
fig.tight_layout()
plt.show()


## 3. Sampled Shapley edge contributions

A positive Shapley value means enabling that edge improves the selected metric on average across sampled edge orderings. It may differ from single-edge removal because Shapley averages the edge's contribution across partial graphs. These are fixed-weight attributions for the selected evaluation split.

In [ ]:
shapley_metric = SELECTION_METRIC
shapley_part = shapley[shapley.metric == shapley_metric].copy()
for model, frame in shapley_part.groupby("model"):
    frame = frame.sort_values("shapley_value", ascending=False)
    print(model, "? Shapley for", shapley_metric)
    display(frame[["edge_id", "source_concept", "target_concept", "shapley_value", "permutation_se", "n_permutations"]].head(20).round(5))

if not shapley_part.empty:
    ax = shapley_part.pivot(index="edge_id", columns="model", values="shapley_value").plot(kind="bar", figsize=(max(9, len(shapley_part.edge_id.unique())*0.45), 4), title=f"Validation Shapley values: {shapley_metric}")
    ax.set_ylabel("Average metric gain from enabling edge"); ax.axhline(0, color="black", lw=.8); plt.xticks(rotation=70, ha="right"); plt.tight_layout(); plt.show()

## 4. Export the top removal candidates for retraining

Selection is done from **validation edge-removal drops**. Choose the criterion above before exporting. The output contains the top `TOP_N` edges per model; the generator turns each row into a DAG with that edge removed and a matching fixed-backbone CREAM training config.

In [ ]:
if selected_view.empty:
    raise ValueError(f"No validation rows found for SELECTION_METRIC={SELECTION_METRIC!r}")
selected = (selected_view.sort_values(["model", "metric_drop_from_removal"], ascending=[True, False])
            .groupby("model", as_index=False).head(TOP_N).copy())
selected["selection_metric"] = SELECTION_METRIC
selected["selection_rank"] = selected.groupby("model")["metric_drop_from_removal"].rank(method="first", ascending=False).astype(int)
selected = selected[["model", "edge_id", "source_index", "target_index", "source_concept", "target_concept", "selection_metric", "selection_rank", "metric_drop_from_removal"]]
selected.to_csv(SELECTION_CSV, index=False)
print("Saved retraining selections:", SELECTION_CSV)
display(selected)

## Interpretation guide

- A large single-edge drop means the trained model currently depends on that edge when the other learned weights are held fixed.
- A large positive Shapley value means the edge helps on average across the sampled partial graphs. A small single-edge drop but larger Shapley value can indicate redundancy with other edges.
- If retraining later recovers performance, the trainable parts adapted to compensate for the removed connection. Compare retrained models against each model's own original baseline.
- The selected edges are metric-dependent. Change `SELECTION_METRIC` and rerun the export cell if the supervisor wants the top edges for concept accuracy or intervention accuracy instead of task accuracy.